# 02 - Macro Regimes

Segundo laboratorio de MacroQuant Lab.

Objetivo: cruzar retornos mensuales de activos con variables macroeconomicas y estudiar que activos funcionan mejor en distintos regimenes.

## 1. Setup

Seguimos la misma filosofia que en el primer notebook: el codigo reutilizable vive en `src/` y el notebook se usa para investigar, visualizar y explicar.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from macroquant_lab.config import CORE_ASSET_TICKERS, FRED_SERIES
from macroquant_lab.data_loader import download_fred_series, download_prices, monthly_prices
from macroquant_lab.macro import build_macro_features, classify_macro_regimes, summarize_returns_by_regime
from macroquant_lab.metrics import calculate_returns
from macroquant_lab.visualization import plot_macro_series, plot_regime_asset_returns

CORE_ASSET_TICKERS, FRED_SERIES

## 2. Descargar datos

Usamos precios de Yahoo Finance y datos macro de FRED. Trabajaremos a frecuencia mensual porque las variables macro no se publican diariamente.

In [ ]:
START_DATE = "2015-01-01"

prices = download_prices(CORE_ASSET_TICKERS, start=START_DATE)
raw_macro = download_fred_series(FRED_SERIES, start=START_DATE)

prices.tail(), raw_macro.tail()

## 3. Crear variables macro

Transformamos series originales en indicadores mas interpretables:

- `Inflation YoY`: variacion interanual del CPI.
- `GDP YoY`: crecimiento interanual del PIB real.
- `Yield Curve 10Y-2Y`: pendiente de la curva de tipos.
- `Real Fed Funds Proxy`: tipo oficial menos inflacion.
- `VIX`: volatilidad esperada del S&P 500.

In [ ]:
macro_features = build_macro_features(raw_macro)
macro_features.tail()

In [ ]:
plot_macro_series(
    macro_features,
    ["Inflation YoY", "Fed Funds Rate", "Yield Curve 10Y-2Y", "GDP YoY", "VIX"],
)

## 4. Clasificar regimenes

Empezamos con reglas simples y transparentes. No son el modelo final; son una primera forma de convertir economia en categorias analizables.

- Inflacion alta: inflacion YoY >= 3%.
- Tipos altos: Fed Funds >= 3%.
- Crecimiento fuerte: PIB YoY >= 2.5%.
- Curva invertida: 10Y - 2Y < 0.

In [ ]:
regimes = classify_macro_regimes(macro_features)
regimes[["macro_regime", "inflation_regime", "rate_regime", "growth_regime", "curve_regime"]].tail(12)

In [ ]:
regimes["macro_regime"].value_counts()

## 5. Retornos mensuales por regimen

Alineamos precios a fin de mes y calculamos retornos mensuales. Despues agrupamos esos retornos por regimen macro.

In [ ]:
monthly_returns = calculate_returns(monthly_prices(prices))
regime_summary = summarize_returns_by_regime(monthly_returns, regimes)
regime_summary.head(15)

In [ ]:
plot_regime_asset_returns(regime_summary)

## 6. Primeras preguntas de research

Intenta responder por escrito:

1. Que activos han funcionado mejor en inflacion alta y tipos altos?
2. Que activo reduce mejor el drawdown cuando la curva esta invertida?
3. Como se comportan bonos de largo plazo cuando los tipos estan altos?
4. Oro diversifica mas en inflacion alta o en crecimiento debil?
5. Hay suficientes meses en cada regimen para confiar en la conclusion?

La ultima pregunta es clave: en finanzas, una tabla bonita con pocas observaciones puede contar una historia demasiado fragil.